In [6]:
from torch import nn
import torch


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weights = nn.Parameter(torch.ones(dim))
    def forward(self, X):
        rms = torch.rsqrt(X.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return X * rms * self.weight

In [8]:
import torch.nn.functional as F

class SwiGLU(nn.Module):
    def __init__(self, d_model, ffn_dim):
        super().__init__()
        self.w_gate = nn.Linear(d_model, ffn_dim, bias=False)
        self.w_up = nn.Linear(d_model, ffn_dim, bias=False)
        self.w_down = nn.Linear(ffn_dim, d_model, bias=False)
    def forward(self, X):
        gate = F.silu(self.w_gate(X))
        up = self.w_up(X)
        down = self.w_down(gate * up)
        return down
    

### Just reminder not actually used in DS 4.1

In [9]:
import math
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_heads
        self.num_queries_per_kv = num_heads // num_kv_heads 

        self.q_proj = nn.Linear(d_model, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(num_heads * self.head_dim, d_model, bias=False)
        
    def forward(self, X, kv_cache=None):
        # X -> [B, T_q, D]
        B, T_q, D = X.shape
        q     = self.q_proj(X).view(B, T_q, self.num_heads, self.head_dim).transpose(1, 2)
        k_new = self.k_proj(X).view(B, T_q, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v_new = self.v_proj(X).view(B, T_q, self.num_kv_heads, self.head_dim).transpose(1, 2)

        if kv_cache is not None:
            k_prev, v_prev = kv_cache
            k = torch.cat([k_prev, k_new], dim=2)
            v = torch.cat([v_prev, v_new], dim=2)
        else:
            k, v = k_new, v_new
        new_kv_cache = (k, v)
        if self.num_queries_per_kv > 1:
            # [B, H_kv, T_total, d_h] -> [B, H, T_total, d_h]
            k_exec = torch.repeat_interleave(k, dim=1, repeats=self.num_queries_per_kv)
            v_exec = torch.repeat_interleave(v, dim=1, repeats=self.num_queries_per_kv)
        else:
            k_exec, v_exec = k, v

        scores = torch.matmul(q, k_exec.transpose(-1, -2)) / math.sqrt(self.head_dim)
        T_total = k.size(2)
        T_past = T_total - T_q
        if T_q > 1:
            causal_mask = torch.triu(torch.full((T_q, T_total), float('-inf'), device=X.device),
                diagonal=T_past + 1)
            scores = scores + causal_mask.unsqueeze(0).unsqueeze(0)
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, v_exec).transpose(1, 2).contiguous().view(B, T_q, -1)
        out = self.out_proj(context)
        return out, new_kv_cache

In [10]:
B, T_q, D = 2, 8, 256
num_heads = 8
num_kv_heads = 2

attn = CausalSelfAttention(d_model=D, num_heads=num_heads, num_kv_heads=num_kv_heads)
x = torch.randn(B, T_q, D)

In [11]:
out, cache = attn(x)

In [13]:
cache

(tensor([[[[ 4.7039e-01, -8.2205e-01, -5.1421e-01,  ...,  3.2619e-01,
             2.3062e-01, -8.1123e-01],
           [-5.9977e-01, -3.9056e-01,  6.0031e-02,  ...,  9.3731e-02,
            -2.8426e-01,  5.3330e-02],
           [-3.5023e-01,  2.3575e-01, -9.4673e-01,  ...,  1.4065e-01,
            -1.2392e-03, -1.4369e-01],
           ...,
           [-1.2985e+00,  4.4047e-01, -2.3368e-01,  ...,  9.9429e-01,
            -8.4147e-01, -1.8797e-01],
           [-4.4365e-01, -2.0237e+00, -9.8400e-01,  ...,  9.6956e-02,
            -5.1668e-02, -9.1504e-01],
           [-5.2408e-02, -5.8298e-02, -5.3419e-01,  ..., -1.1103e+00,
            -1.5516e-01, -7.1177e-01]],
 
          [[ 3.7887e-01, -3.8862e-02,  1.6357e+00,  ...,  1.0332e+00,
             1.7480e-01, -4.8301e-02],
           [ 5.2389e-03, -2.5922e-01, -5.2022e-01,  ..., -4.9736e-02,
             8.2594e-01,  1.4220e+00],
           [ 7.6345e-02,  4.6009e-02,  5.9408e-01,  ...,  2.3153e-01,
             8.7798e-01, -3.0988e-01],


In [31]:
class CausalEncoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        ...

### This one on the other hand is used 

In [36]:
class Embeddings(nn.Module):
    def __init__(self, emb_dim, vocab_size,):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)

    def forward(self, X):
        return self.embed(X)

In [37]:
class ResidualStreams(nn.Module):
    def __init__(self, streams = 4,vocab_size = 129280, emb_dim = 5120):
        super().__init__()
        self.embed = Embeddings(emb_dim, vocab_size)
        self.streams = streams

    def forward(self, token_ids):
        h = self.embed(token_ids)
        return h.unsqueeze(2).repeat(1, 1, self.streams, 1)

In [ ]:
vocab_size = 129280
emb_dim = 5120
embed = nn.Embedding(vocab_size, emb_dim)

In [34]:
X = torch.LongTensor([[0, 2, 0, 5]])
B,T  =X.shape

embed(X).unsqueeze(2).repeat(1, 1, 4, 1).shape

torch.Size([1, 4, 4, 5120])